In [0]:
# ============================================================
# EHIP V2 - CONFIGURATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "project_ehip"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

BRONZE_DOCTOR = f"{CATALOG}.{BRONZE_SCHEMA}.doctor"
BRONZE_PATIENT = f"{CATALOG}.{BRONZE_SCHEMA}.patient"
BRONZE_HISTORY = f"{CATALOG}.{BRONZE_SCHEMA}.medical_history"

SILVER_DOCTOR = f"{CATALOG}.{SILVER_SCHEMA}.doctor"
SILVER_PATIENT = f"{CATALOG}.{SILVER_SCHEMA}.patient"
SILVER_HISTORY = f"{CATALOG}.{SILVER_SCHEMA}.medical_history"

GOLD_PATIENT_HEALTH = f"{CATALOG}.{GOLD_SCHEMA}.patient_health_analytics"
GOLD_DOCTOR_SUMMARY = f"{CATALOG}.{GOLD_SCHEMA}.doctor_patient_summary"
GOLD_HOSPITAL_KPI = f"{CATALOG}.{GOLD_SCHEMA}.hospital_kpis"

print("EHIP V2 configuration loaded.")

# ============================================================
# READ BRONZE
# ============================================================

doctor_bronze = spark.table(BRONZE_DOCTOR)

patient_bronze = spark.table(BRONZE_PATIENT)

history_bronze = spark.table(BRONZE_HISTORY)

print("Bronze datasets loaded.")

EHIP V2 configuration loaded.
Bronze datasets loaded.


In [0]:
# ============================================================
# STANDARDIZE SOURCE DATA TYPES
# ============================================================

doctor_bronze = doctor_bronze.withColumn(
    "UpdatedAt",
    F.to_timestamp("UpdatedAt")
)

patient_bronze = patient_bronze.withColumn(
    "UpdatedAt",
    F.to_timestamp("UpdatedAt")
)

history_bronze = history_bronze.withColumn(
    "UpdatedAt",
    F.to_timestamp("UpdatedAt")
)

history_bronze = (
    history_bronze
    .withColumn("SugarLevel", F.col("SugarLevel").cast("double"))
    .withColumn("BloodPressureSys", F.col("BloodPressureSys").cast("int"))
    .withColumn("BloodPressureDia", F.col("BloodPressureDia").cast("int"))
    .withColumn("HeartRate", F.col("HeartRate").cast("int"))
    .withColumn("Cholesterol", F.col("Cholesterol").cast("int"))
    .withColumn("BMI", F.col("BMI").cast("double"))
)

In [0]:
# ============================================================
# DATA QUALITY - DOCTOR
# ============================================================

doctor_invalid = doctor_bronze.filter(
    F.col("DoctorId").isNull()
    |
    F.col("FirstName").isNull()
    |
    F.col("LastName").isNull()
    |
    F.col("Email").isNull()
    |
    F.col("UpdatedAt").isNull()
)
doctor_invalid_count = doctor_invalid.count()

print(
    f"Invalid Doctor records: {doctor_invalid_count}"
)

doctor_duplicates = (
    doctor_bronze
    .groupBy("DoctorId")
    .count()
    .filter(F.col("count") > 1)
)

doctor_duplicate_count = doctor_duplicates.count()

print(
    f"Duplicate Doctor IDs: {doctor_duplicate_count}"
)


Invalid Doctor records: 0
Duplicate Doctor IDs: 0


In [0]:
# ============================================================
# DATA QUALITY - PATIENT
# ============================================================

patient_invalid = patient_bronze.filter(
    F.col("PatientId").isNull()
    |
    F.col("DoctorId").isNull()
    |
    F.col("FirstName").isNull()
    |
    F.col("LastName").isNull()
    |
    F.col("DateOfBirth").isNull()
    |
    F.col("UpdatedAt").isNull()
)
patient_duplicates = (
    patient_bronze
    .groupBy("PatientId")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Invalid Patient records:",
    patient_invalid.count()
)

print(
    "Duplicate Patient IDs:",
    patient_duplicates.count()
)


Invalid Patient records: 0
Duplicate Patient IDs: 0


In [0]:
# ============================================================
# DATA QUALITY - MEDICAL HISTORY
# ============================================================

history_invalid = history_bronze.filter(
    F.col("HistoryId").isNull()
    |
    F.col("PatientId").isNull()
    |
    F.col("UpdatedAt").isNull()
    |
    (F.col("BMI") <= 0)
    |
    (F.col("HeartRate") <= 0)
    |
    (F.col("SugarLevel") < 0)
    |
    (F.col("BloodPressureSys") <= 0)
    |
    (F.col("BloodPressureDia") <= 0)
)
print(
    "Invalid MedicalHistory records:",
    history_invalid.count()
)

Invalid MedicalHistory records: 0


In [0]:
# ============================================================
# REFERENTIAL INTEGRITY
# ============================================================

invalid_patient_doctor = (
    patient_bronze.alias("p")
    .join(
        doctor_bronze.select("DoctorId").distinct().alias("d"),
        F.col("p.DoctorId") == F.col("d.DoctorId"),
        "left"
    )
    .filter(F.col("d.DoctorId").isNull())
)

print(
    "Invalid Patient → Doctor references:",
    invalid_patient_doctor.count()
)

invalid_history_patient = (
    history_bronze.alias("h")
    .join(
        patient_bronze.select("PatientId").distinct().alias("p"),
        F.col("h.PatientId") == F.col("p.PatientId"),
        "left"
    )
    .filter(F.col("p.PatientId").isNull())
)

print(
    "Invalid MedicalHistory → Patient references:",
    invalid_history_patient.count()
)

Invalid Patient → Doctor references: 0
Invalid MedicalHistory → Patient references: 0


In [0]:
# ============================================================
# DATA QUALITY SUMMARY
# ============================================================

dq_summary = spark.createDataFrame(
    [
        (
            "Doctor",
            doctor_bronze.count(),
            doctor_invalid_count,
            doctor_duplicate_count
        ),
        (
            "Patient",
            patient_bronze.count(),
            patient_invalid.count(),
            patient_duplicates.count()
        ),
        (
            "MedicalHistory",
            history_bronze.count(),
            history_invalid.count(),
            0
        )
    ],
    [
        "TableName",
        "TotalRecords",
        "InvalidRecords",
        "DuplicateRecords"
    ]
)

dq_summary = dq_summary.withColumn(
    "Status",
    F.when(
        (F.col("InvalidRecords") == 0) &
        (F.col("DuplicateRecords") == 0),
        "PASS"
    ).otherwise("FAIL")
)

dq_summary.show()

+--------------+------------+--------------+----------------+------+
|     TableName|TotalRecords|InvalidRecords|DuplicateRecords|Status|
+--------------+------------+--------------+----------------+------+
|        Doctor|         500|             0|               0|  PASS|
|       Patient|       10000|             0|               0|  PASS|
|MedicalHistory|       40000|             0|               0|  PASS|
+--------------+------------+--------------+----------------+------+



In [0]:
dq_summary.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{CATALOG}.gold.data_quality_metrics"
    )

In [0]:
# ============================================================
# INCREMENTAL LOAD HELPER
# ============================================================

def get_incremental_data(
    source_df,
    target_table,
    updated_column,
    primary_key
):

    if not spark.catalog.tableExists(target_table):

        print(
            f"{target_table} does not exist."
        )

        print(
            "Running initial full load."
        )

        return source_df

    target_df = spark.table(target_table)

    watermark = (
        target_df
        .select(
            F.max(updated_column).alias("max_updated")
        )
        .collect()[0]["max_updated"]
    )

    if watermark is None:

        print(
            "No watermark found. Running full load."
        )

        return source_df

    print(
        f"Watermark for {target_table}: {watermark}"
    )

    incremental_df = source_df.filter(
        F.col(updated_column) >= F.lit(watermark)
    )

    print(
        f"Incremental {primary_key} records:"
        f" {incremental_df.count()}"
    )

    return incremental_df

In [0]:
doctor_incremental = get_incremental_data(
    doctor_bronze,
    SILVER_DOCTOR,
    "UpdatedAt",
    "DoctorId"
)

doctor_incremental = (
    doctor_incremental

    .withColumn(
        "FirstName",
        F.initcap(F.trim("FirstName"))
    )

    .withColumn(
        "LastName",
        F.initcap(F.trim("LastName"))
    )

    .withColumn(
        "DoctorFullName",
        F.concat_ws(
            " ",
            F.col("FirstName"),
            F.col("LastName")
        )
    )

    .withColumn(
        "Specialization",
        F.initcap(F.trim("Specialization"))
    )

    .withColumn(
        "Email",
        F.lower(F.trim("Email"))
    )

    .withColumn(
        "Phone",
        F.trim("Phone")
    )
)

Watermark for project_ehip.silver.doctor: 2026-04-19 02:26:00
Incremental DoctorId records: 1


In [0]:
# ============================================================
# INCREMENTAL MERGE - DOCTOR
# ============================================================

if not spark.catalog.tableExists(SILVER_DOCTOR):

    (
        doctor_incremental
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_DOCTOR)
    )

else:

    doctor_delta = DeltaTable.forName(
        spark,
        SILVER_DOCTOR
    )

    (
        doctor_delta.alias("target")

        .merge(
            doctor_incremental.alias("source"),
            "target.DoctorId = source.DoctorId"
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

print("Doctor incremental merge completed.")

Doctor incremental merge completed.


In [0]:
# ============================================================
# INCREMENTAL LOAD & MERGE - PATIENT
# ============================================================

patient_incremental = get_incremental_data(
    patient_bronze,
    SILVER_PATIENT,
    "UpdatedAt",
    "PatientId"
)
patient_incremental = (
    patient_incremental

    .withColumn(
        "FirstName",
        F.initcap(F.trim("FirstName"))
    )

    .withColumn(
        "LastName",
        F.initcap(F.trim("LastName"))
    )

    .withColumn(
        "PatientFullName",
        F.concat_ws(
            " ",
            F.col("FirstName"),
            F.col("LastName")
        )
    )

    .withColumn(
        "Gender",
        F.lower(F.trim("Gender"))
    )

    .withColumn(
        "Age",
        F.floor(
            F.datediff(
                F.current_date(),
                F.col("DateOfBirth")
            ) / F.lit(365.25)
        )
    )

    .withColumn(
        "AgeGroup",

        F.when(F.col("Age") < 18, "Under 18")

        .when(
            (F.col("Age") >= 18) &
            (F.col("Age") <= 29),
            "18-29"
        )

        .when(
            (F.col("Age") >= 30) &
            (F.col("Age") <= 44),
            "30-44"
        )

        .when(
            (F.col("Age") >= 45) &
            (F.col("Age") <= 59),
            "45-59"
        )

        .otherwise("60+")
    )
)
if not spark.catalog.tableExists(SILVER_PATIENT):

    (
        patient_incremental
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_PATIENT)
    )

else:

    patient_delta = DeltaTable.forName(
        spark,
        SILVER_PATIENT
    )

    (
        patient_delta.alias("target")

        .merge(
            patient_incremental.alias("source"),
            "target.PatientId = source.PatientId"
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

print("Patient incremental merge completed.")

Watermark for project_ehip.silver.patient: 2026-08-27 06:55:01
Incremental PatientId records: 1
Patient incremental merge completed.


In [0]:
# ============================================================
# INCREMENTAL LOAD & MERGE - MEDICAL HISTORY
# ============================================================

history_incremental = get_incremental_data(
    history_bronze,
    SILVER_HISTORY,
    "UpdatedAt",
    "HistoryId"
)
history_incremental = (
    history_incremental

    .withColumn(
        "BMI_Category",

        F.when(
            F.col("BMI") < 18.5,
            "Underweight"
        )

        .when(
            F.col("BMI") < 25,
            "Normal"
        )

        .when(
            F.col("BMI") < 30,
            "Overweight"
        )

        .otherwise("Obese")
    )

    .withColumn(
        "BloodPressure_Category",

        F.when(
            (F.col("BloodPressureSys") < 120) &
            (F.col("BloodPressureDia") < 80),
            "Normal"
        )

        .when(
            (F.col("BloodPressureSys") < 130) &
            (F.col("BloodPressureDia") < 80),
            "Elevated"
        )

        .when(
            (F.col("BloodPressureSys") < 140) |
            (F.col("BloodPressureDia") < 90),
            "High Stage 1"
        )

        .otherwise("High Stage 2")
    )

    .withColumn(
        "Cholesterol_Category",

        F.when(
            F.col("Cholesterol") < 200,
            "Normal"
        )

        .when(
            F.col("Cholesterol") <= 239,
            "Borderline High"
        )

        .otherwise("High")
    )

    .withColumn(
        "Sugar_Category",

        F.when(
            F.col("SugarLevel") < 70,
            "Low"
        )

        .when(
            F.col("SugarLevel") <= 99,
            "Normal"
        )

        .when(
            F.col("SugarLevel") <= 125,
            "Prediabetic Range"
        )

        .otherwise("High")
    )

    .withColumn(
        "Smoking_Status",

        F.when(
            F.col("Smoking"),
            "Smoker"
        )

        .otherwise("Non-Smoker")
    )

    .withColumn(
        "Alcohol_Status",

        F.when(
            F.col("AlcoholConsumption"),
            "Consumes Alcohol"
        )

        .otherwise("No Alcohol")
    )
)

if not spark.catalog.tableExists(SILVER_HISTORY):

    (
        history_incremental
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_HISTORY)
    )

else:

    history_delta = DeltaTable.forName(
        spark,
        SILVER_HISTORY
    )

    (
        history_delta.alias("target")

        .merge(
            history_incremental.alias("source"),
            "target.HistoryId = source.HistoryId"
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

print(
    "MedicalHistory incremental merge completed."
)

Watermark for project_ehip.silver.medical_history: 2026-09-02 02:00:00
Incremental HistoryId records: 1
MedicalHistory incremental merge completed.


In [0]:
doctor_silver = spark.table(SILVER_DOCTOR)
patient_silver = spark.table(SILVER_PATIENT)
history_silver = spark.table(SILVER_HISTORY)

In [0]:
# ============================================================
# GOLD - PATIENT HEALTH ANALYTICS
# ============================================================

patient_health_gold = (
    history_silver.alias("h")

    .join(
        patient_silver.alias("p"),
        F.col("h.PatientId") == F.col("p.PatientId"),
        "inner"
    )

    .join(
        doctor_silver.alias("d"),
        F.col("p.DoctorId") == F.col("d.DoctorId"),
        "left"
    )

    .select(
        F.col("h.HistoryId"),

        F.col("p.PatientId"),
        F.col("p.PatientFullName"),
        F.col("p.Age"),
        F.col("p.AgeGroup"),
        F.col("p.Gender"),

        F.col("d.DoctorId"),
        F.col("d.DoctorFullName"),
        F.col("d.Specialization"),

        F.col("h.RecordedAt"),

        F.col("h.SugarLevel"),
        F.col("h.HasDiabetes"),

        F.col("h.BloodPressureSys"),
        F.col("h.BloodPressureDia"),

        F.col("h.HeartRate"),
        F.col("h.Cholesterol"),
        F.col("h.BMI"),

        F.col("h.Smoking"),
        F.col("h.AlcoholConsumption"),

        F.col("h.BMI_Category"),
        F.col("h.BloodPressure_Category"),
        F.col("h.Cholesterol_Category"),
        F.col("h.Sugar_Category"),

        F.col("h.Smoking_Status"),
        F.col("h.Alcohol_Status")
    )
)
(
    patient_health_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "project_ehip.gold.patient_health_analytics"
    )
)

print("Gold Patient Health Analytics created.")


doctor_patient_gold = (
    patient_silver.alias("p")

    .join(
        doctor_silver.alias("d"),
        F.col("p.DoctorId") == F.col("d.DoctorId"),
        "inner"
    )

    .groupBy(
        F.col("d.DoctorId"),
        F.col("d.DoctorFullName"),
        F.col("d.Specialization")
    )

    .agg(
        F.countDistinct(
            F.col("p.PatientId")
        ).alias("PatientCount"),

        F.round(
            F.avg(F.col("p.Age")),
            2
        ).alias("AveragePatientAge")
    )
)
(
    doctor_patient_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "project_ehip.gold.doctor_patient_summary"
    )
)
print("Gold Doctor Patient Summary created.")

hospital_kpi_gold = (
    patient_health_gold
    .agg(
        F.countDistinct("PatientId")
        .alias("TotalPatients"),

        F.countDistinct("DoctorId")
        .alias("TotalDoctors"),

        F.count("HistoryId")
        .alias("TotalMedicalRecords"),

        F.round(
            F.avg("Age"),
            2
        ).alias("AveragePatientAge"),

        F.round(
            F.avg("BMI"),
            2
        ).alias("AverageBMI"),

        F.round(
            F.avg("SugarLevel"),
            2
        ).alias("AverageSugarLevel"),

        F.sum(
            F.when(
                F.col("HasDiabetes"),
                1
            ).otherwise(0)
        ).alias("DiabeticRecords"),

        F.sum(
            F.when(
                F.col("BloodPressure_Category")
                .isin("High Stage 1", "High Stage 2"),
                1
            ).otherwise(0)
        ).alias("HighBPRecords"),

        F.sum(
            F.when(
                F.col("Cholesterol_Category") == "High",
                1
            ).otherwise(0)
        ).alias("HighCholesterolRecords")
    )
)
(
    hospital_kpi_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "project_ehip.gold.hospital_kpis"
    )
)
print("Gold Hospital KPIs created.")



print("========== EHIP V2 FINAL VALIDATION ==========")

print(
    "Bronze Doctor:",
    doctor_bronze.count()
)

print(
    "Bronze Patient:",
    patient_bronze.count()
)

print(
    "Bronze Medical History:",
    history_bronze.count()
)

print(
    "Silver Doctor:",
    spark.table(
        "project_ehip.silver.doctor"
    ).count()
)

print(
    "Silver Patient:",
    spark.table(
        "project_ehip.silver.patient"
    ).count()
)

print(
    "Silver Medical History:",
    spark.table(
        "project_ehip.silver.medical_history"
    ).count()
)

print(
    "Gold Patient Health:",
    spark.table(
        "project_ehip.gold.patient_health_analytics"
    ).count()
)



Gold Patient Health Analytics created.
Gold Doctor Patient Summary created.
Gold Hospital KPIs created.
========== EHIP V2 FINAL VALIDATION ==========
Bronze Doctor: 500
Bronze Patient: 10000
Bronze Medical History: 40000
Silver Doctor: 500
Silver Patient: 10000
Silver Medical History: 40000
Gold Patient Health: 40000


In [0]:
# ============================================================
# PIPELINE METRICS
# ============================================================

pipeline_metrics = spark.createDataFrame(
    [
        (
            "Doctor",
            doctor_incremental.count()
        ),
        (
            "Patient",
            patient_incremental.count()
        ),
        (
            "MedicalHistory",
            history_incremental.count()
        )
    ],
    [
        "Entity",
        "IncrementalRecordsProcessed"
    ]
)

pipeline_metrics.show()

+--------------+---------------------------+
|        Entity|IncrementalRecordsProcessed|
+--------------+---------------------------+
|        Doctor|                          1|
|       Patient|                          1|
|MedicalHistory|                          1|
+--------------+---------------------------+



In [0]:
spark.table(SILVER_DOCTOR) \
    .filter(F.col("DoctorId") == 1) \
    .select(
        "DoctorId",
        "Specialization",
        "UpdatedAt"
    ) \
    .show()

+--------+--------------+-------------------+
|DoctorId|Specialization|          UpdatedAt|
+--------+--------------+-------------------+
|       1|  Cardiologist|2024-01-05 01:01:00|
+--------+--------------+-------------------+

